![Week 3 Day 2](../../assets/banners/w3d2.png)

# Week 3 Day 2 - Solution: The Marks Auditor Agent

The exercise: an agent with `sum_criteria` and `list_questions`, a structured `AuditReport`, a custom `@wrap_tool_call` that counts calls per tool, and a `ToolCallLimitMiddleware` capping `sum_criteria` at 5 calls.

⏮ Back to the lab: `week3/day2.ipynb`

In [ ]:
import os
from collections import Counter

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, wrap_tool_call
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from pydantic import BaseModel, Field

from gradewise.config import DATA_DIR
from gradewise.models import chat_model
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK: only the cells marked 🔑 need a key, the rest run offline.")
    print("   To set one up, see Part 5 of the setup guide (and remember to SAVE .env!)")

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}

In [ ]:
@tool
def list_questions() -> str:
    """List every question on the CS201 midterm with its ID and marks."""
    return "\n".join(f"{q.id}: {q.max_marks:g} marks" for q in exam.questions)


@tool(parse_docstring=True)
def sum_criteria(question_id: str) -> str:
    """Add up a question's marking criteria and compare the total with the question's marks.

    Args:
        question_id: The question ID, e.g. "Q3".
    """
    q = QUESTIONS.get(question_id)
    if q is None:
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    verdict = "adds up ✅" if abs(q.criteria_total - q.max_marks) < 0.01 else "DOES NOT add up ⚠️"
    return f"{q.id}: criteria total {q.criteria_total:g}, question worth {q.max_marks:g}: {verdict}"


class AuditReport(BaseModel):
    """The auditor's verdict on the exam's marks."""

    checked: list[str] = Field(description="The question IDs that were checked")
    all_add_up: bool
    problems: list[str] = Field(default_factory=list)


TOOL_CALLS: Counter = Counter()


@wrap_tool_call
def count_tool_calls(request, handler):
    """Count how often each tool is called (the limit middleware may still block some)."""
    TOOL_CALLS[request.tool_call["name"]] += 1
    return handler(request)


def build_auditor(model, response_format=AuditReport):
    return create_agent(
        model,
        tools=[list_questions, sum_criteria],
        system_prompt="You audit exam marks. List the questions, check each one with sum_criteria, then report.",
        response_format=response_format,
        middleware=[
            count_tool_calls,
            ToolCallLimitMiddleware(tool_name="sum_criteria", run_limit=5),
            ModelCallLimitMiddleware(run_limit=12),  # 🚀 PRO: a seatbelt on the whole run
        ],
    )

### 🎭 With a stand-in model
The stand-in tries to check **six** questions. Watch the sixth `sum_criteria` call get blocked by the limit. 💡 The limit middleware stops it *before* the tool runs, so our counter, which wraps the actual tool execution, only counts 5.

In [ ]:
# 🎭 A STAND-IN model, so you can watch agents work with no API key and no cost.
# 💡 GenericFakeChatModel replies with the messages we script, in order. It can't really call
#    tools, so bind_tools() just returns the same model: the tool calls are in our script.
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(GenericFakeChatModel):
    """Replays a script of AIMessages. ⚠️ A teaching prop, not an LLM: it never reads your prompt."""

    def bind_tools(self, tools, **kwargs):
        return self


def scripted(*replies: AIMessage) -> ScriptedModel:
    return ScriptedModel(messages=iter(replies))


def call(name: str, call_id: str = "call_1", **args) -> AIMessage:
    """A scripted AIMessage that asks for ONE tool call, exactly like a real model would."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

In [ ]:
ids = ["Q1", "Q2a", "Q2b", "Q3", "Q4a", "Q4b"]
script = [call("list_questions")]
script += [call("sum_criteria", call_id=f"c{i}", question_id=qid) for i, qid in enumerate(ids)]
script += [
    call(
        "AuditReport",
        call_id="final",
        checked=ids[:5],
        all_add_up=True,
        problems=["Q4b not checked: tool limit reached"],
    )
]

TOOL_CALLS.clear()
result = build_auditor(scripted(*script), response_format=ToolStrategy(AuditReport)).invoke(
    {"messages": [{"role": "user", "content": "Audit the first six questions."}]}
)
for m in result["messages"]:
    if m.type == "tool":
        print("🛠️ ", m.content)
print("\nCalls counted by our middleware:", dict(TOOL_CALLS))
result["structured_response"]

### 🔑 With a real model

In [ ]:
TOOL_CALLS.clear()
result = build_auditor(chat_model(temperature=0)).invoke(
    {"messages": [{"role": "user", "content": "Audit questions Q1 to Q4b."}]}
)
print(dict(TOOL_CALLS))
result["structured_response"]